# FLAG 2027 - Notebook 07: VoxCeleb2 (thousands of people) - pilot + scale-up

Notebook 06 proved that **more training people** moves the same-gender score (38.5 -> 35.6). VoxCeleb2 has ~6,000 people (organizers confirmed external data is allowed, must be declared).

**Per VoxCeleb2 zip (one at a time, resumable):**
1. Download from Hugging Face (uses your `HF_TOKEN` Colab secret).
2. Pick up to `CLIPS_PER_ID` clips per person, spread over different videos.
3. With ffmpeg: audio -> 16 kHz wav, one video frame -> face image (VoxCeleb2 videos are already face crops).
4. **Bengali filter (organizers' rule):** a language-ID model (VoxLingua107) checks every clip; Bengali clips are dropped.
5. Extract CLIP / ECAPA / WavLM (same as our K10 system), save to Drive, delete the zip.

Then train **E2 (challenge + MAV-Celeb)** vs **E3 (+ VoxCeleb2)** with the same honest validation, and build a submission ZIP for E3.

**First run = pilot with the official test zip (8.9 GB, 118 people), about 1-1.5 h.** Later we add the big zips to `ZIPS`.

**When done:** paste the **SUMMARY** to Claude.

In [ ]:
# Cell 1 - Drive, GPU, packages, Hugging Face token
from google.colab import drive, userdata
drive.mount('/content/drive')
!pip -q install speechbrain huggingface_hub
import torch, shutil
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
HF_TOKEN = userdata.get('HF_TOKEN')
print('Device:', dev, '| HF token found:', HF_TOKEN is not None)
total, used, free = shutil.disk_usage('/content')
print('Colab disk free: %.0f GB' % (free / 1e9))

In [ ]:
# Cell 2 - settings: which VoxCeleb2 zips to process (finished zips are skipped automatically)
ZIPS = ['vox2/vox2_test_mp4.zip',          # pilot, done 14 Oct (118 people)
        'vox2/vox2_mp4_1.zip']             # 50 GB, ~1,000 people (re-packed by the uploader)
# later add 'vox2/vox2_mp4_2.zip', 'vox2/vox2_mp4_3.zip', ... one per session
CLIPS_PER_ID = 20                          # big zips: more PEOPLE matter more than clips per person
THREADS = 8                                # parallel ffmpeg jobs
RUN_TRAINING = False                       # extraction only; training with the domain fix is in Notebook 08

In [ ]:
# Cell 3 - paths, helpers, challenge + MAV-Celeb features (as Notebook 06)
import os, re, json, time, zipfile, copy, subprocess
import numpy as np, pandas as pd
import librosa, soundfile as sf
from PIL import Image
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027')
FEATDIR = os.path.join(PROJECT, 'features')
CACHE = os.path.join(PROJECT, 'models_cache')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
LOCAL = '/content/flag_small'
os.makedirs(LOCAL, exist_ok=True)
for z in ['train_set.zip', 'dev_set.zip']:
    flag = os.path.join(LOCAL, z + '.done')
    if not os.path.exists(flag):
        with zipfile.ZipFile(os.path.join(ZIP_DIR, z)) as zf:
            zf.extractall(LOCAL, members=[m for m in zf.namelist() if m.endswith(('.csv', '.txt'))])
        open(flag, 'w').close()
INDEX = [os.path.join(r, f) for r, _, fs in os.walk(LOCAL) for f in fs if f.endswith(('.csv', '.txt'))]
def find_one(*parts):
    hits = [p for p in INDEX if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)

SPLIT_NAMES = ['train', 'no_gender_English', 'no_gender_Bangla', 'gender_English', 'gender_Bangla']
FEAT = {}
Xv, y = read_feat(find_one('train_English_voices.csv'), 192)
FEAT['ecapa'] = {'train': Xv}
for s in SPLIT_NAMES[1:]:
    track, lang = s.rsplit('_', 1)
    FEAT['ecapa'][s] = read_feat(find_one('/' + track + '/', lang + '_test_voices.csv'), 192)[0]
for name in ['clip', 'wavlm_sv']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in SPLIT_NAMES}
y = y.astype(int)
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y)
id_gender = {c: spk2g[cls2spk[c]] for c in ids}
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz')))

SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
print('challenge rows', len(y), '| MAV-Celeb rows', len(MAV['spk']))

In [ ]:
# Cell 4 - encoders: official ECAPA, CLIP, WavLM (as Notebook 06) + language-ID for the Bengali filter
try:
    from speechbrain.inference.interfaces import Pretrained
    from speechbrain.inference.classifiers import EncoderClassifier
except ImportError:
    from speechbrain.pretrained.interfaces import Pretrained
    from speechbrain.pretrained import EncoderClassifier

class Encoder(Pretrained):
    MODULES_NEEDED = ['compute_features', 'mean_var_norm', 'embedding_model']
    def encode_batch(self, wavs, wav_lens=None):
        if len(wavs.shape) == 1:
            wavs = wavs.unsqueeze(0)
        if wav_lens is None:
            wav_lens = torch.ones(wavs.shape[0], device=self.device)
        wavs, wav_lens = wavs.to(self.device).float(), wav_lens.to(self.device)
        feats = self.mods.compute_features(wavs)
        feats = self.mods.mean_var_norm(feats, wav_lens)
        return self.mods.embedding_model(feats, wav_lens)

ECAPA = Encoder.from_hparams(source='yangwang825/ecapa-tdnn-vox2', savedir=os.path.join(CACHE, 'ecapa_vox2'), run_opts={'device': dev})
ECAPA.eval()
LID = EncoderClassifier.from_hparams(source='speechbrain/lang-id-voxlingua107-ecapa', savedir=os.path.join(CACHE, 'lid_voxlingua107'), run_opts={'device': dev})

def batches_by_length(paths, bs):
    order = np.argsort([sf.info(p).duration for p in paths])
    for i in range(0, len(order), bs):
        yield order[i:i + bs]

def pad(paths, idx, max_sec=None):
    wavs = [librosa.load(paths[j], sr=16000, mono=True)[0] for j in idx]
    if max_sec:
        wavs = [w[:16000 * max_sec] for w in wavs]
    lens = torch.tensor([len(w) for w in wavs], dtype=torch.float); L = int(lens.max())
    b = torch.zeros(len(wavs), L)
    for k, w in enumerate(wavs):
        b[k, :len(w)] = torch.from_numpy(w)
    return b, lens / L

def enc_ecapa(paths, bs=16):
    out = np.zeros((len(paths), 192), np.float32)
    for idx in tqdm(list(batches_by_length(paths, bs)), desc='ecapa'):
        b, rl = pad(paths, idx)
        with torch.no_grad():
            out[idx] = ECAPA.encode_batch(b, rl).squeeze(1).cpu().numpy()
    return out

def language_of(paths, bs=32):
    # returns ISO code of the most likely language per clip (e.g. 'en', 'bn', 'hi')
    out = [''] * len(paths)
    for idx in tqdm(list(batches_by_length(paths, bs)), desc='language-ID'):
        b, rl = pad(paths, idx, max_sec=10)
        with torch.no_grad():
            labs = LID.classify_batch(b.to(dev), rl.to(dev))[3]
        for j, lab in zip(idx, labs):
            out[j] = lab.split(':')[0].strip()
    return np.array(out)

def enc_clip(paths):
    from transformers import CLIPModel, CLIPProcessor
    name = 'openai/clip-vit-large-patch14'
    model = CLIPModel.from_pretrained(name).half().to(dev).eval(); proc = CLIPProcessor.from_pretrained(name)
    out = []
    for i in tqdm(range(0, len(paths), 64), desc='clip'):
        ims = [Image.open(p).convert('RGB') for p in paths[i:i + 64]]
        x = proc(images=ims, return_tensors='pt')['pixel_values'].to(dev, torch.float16)
        with torch.no_grad():
            out.append(model.visual_projection(model.vision_model(pixel_values=x).pooler_output).float().cpu().numpy())
    return np.concatenate(out)

def enc_wavlm(paths, max_sec=10):
    from transformers import AutoFeatureExtractor, WavLMForXVector
    name = 'microsoft/wavlm-base-plus-sv'
    fe = AutoFeatureExtractor.from_pretrained(name); model = WavLMForXVector.from_pretrained(name).to(dev).eval()
    out = []
    for i in tqdm(range(0, len(paths), 32), desc='wavlm'):
        wavs = [librosa.load(p, sr=16000, mono=True)[0][:16000 * max_sec] for p in paths[i:i + 32]]
        x = fe(wavs, sampling_rate=16000, padding=True, return_attention_mask=True, return_tensors='pt').to(dev)
        with torch.no_grad():
            out.append(model(**x).embeddings.cpu().numpy())
    return np.concatenate(out)
print('encoders ready')

In [ ]:
# Cell 5 - VoxCeleb2 gender metadata (useful later for gender-aware training)
from huggingface_hub import hf_hub_download
VOX_GENDER = {}
try:
    mp = hf_hub_download('ProgramComputer/voxceleb', 'vox2/vox2_meta.csv', repo_type='dataset', token=HF_TOKEN, local_dir='/content/vox')
    vm = pd.read_csv(mp, sep=None, engine='python')
    vm.columns = [c.strip().lower() for c in vm.columns]
    idc = [c for c in vm.columns if 'voxceleb2' in c or c.startswith('id')][0]
    gc = [c for c in vm.columns if 'gender' in c][0]
    VOX_GENDER = dict(zip(vm[idc].astype(str).str.strip(), vm[gc].astype(str).str.strip().str.lower()))
    note('VoxCeleb2 meta: %d people with gender labels' % len(VOX_GENDER))
except Exception as e:
    note('VoxCeleb2 meta not loaded (%r) - continuing without gender labels' % e)

In [ ]:
# Cell 6 - process each VoxCeleb2 zip: download -> pick clips -> ffmpeg -> Bengali filter -> features (RESUMABLE)
def ffmpeg_one(job):
    mp4, wav, jpg = job
    a = subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-i', mp4, '-vn', '-ac', '1', '-ar', '16000', wav]).returncode
    v = subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-ss', '1.0', '-i', mp4, '-frames:v', '1', '-q:v', '2', jpg]).returncode
    return a == 0 and v == 0 and os.path.exists(wav) and os.path.exists(jpg)

for zname in ZIPS:
    tag = os.path.basename(zname).replace('.zip', '')
    out_path = os.path.join(FEATDIR, 'vox_' + tag + '.npz')
    if os.path.exists(out_path):
        print(tag, 'already processed - skipping'); continue
    t0 = time.time()
    zpath = hf_hub_download('ProgramComputer/voxceleb', zname, repo_type='dataset', token=HF_TOKEN, local_dir='/content/vox')
    print('%s downloaded in %.1f min' % (tag, (time.time() - t0) / 60))
    try:
        zf = zipfile.ZipFile(zpath)
    except zipfile.BadZipFile:
        note('%s is NOT a valid standalone zip - skipped' % tag); os.remove(zpath); continue
    # group clips: person -> video -> clips
    tree = {}
    for n in zf.namelist():
        mm = re.search(r'(id\d{5})/([^/]+)/([^/]+\.mp4)$', n)
        if mm:
            tree.setdefault(mm.group(1), {}).setdefault(mm.group(2), []).append(n)
    rng = np.random.default_rng(0)
    picked = []   # (person, member)
    for pid, vids in tree.items():
        pools = [list(rng.permutation(v)) for v in vids.values()]
        chosen = []
        while len(chosen) < CLIPS_PER_ID and any(pools):
            for p in pools:            # round-robin over videos
                if p and len(chosen) < CLIPS_PER_ID:
                    chosen.append(p.pop())
        picked += [(pid, c) for c in chosen]
    note('%s: %d people, %d clips picked' % (tag, len(tree), len(picked)))
    W = '/content/vox_work/' + tag; os.makedirs(W, exist_ok=True)
    jobs = []
    for i, (pid, member) in enumerate(picked):
        mp4 = zf.extract(member, W + '/mp4')
        jobs.append((mp4, '%s/%06d.wav' % (W, i), '%s/%06d.jpg' % (W, i)))
    zf.close(); os.remove(zpath)            # free the disk early
    with ThreadPoolExecutor(THREADS) as ex:
        ok = list(tqdm(ex.map(ffmpeg_one, jobs), total=len(jobs), desc='ffmpeg'))
    keep = [i for i, good in enumerate(ok) if good]
    wavs = [jobs[i][1] for i in keep]; jpgs = [jobs[i][2] for i in keep]; spk = [picked[i][0] for i in keep]
    lang = language_of(wavs)
    not_bn = lang != 'bn'
    note('%s: ffmpeg ok %d/%d | Bengali clips removed: %d | top languages: %s'
         % (tag, len(keep), len(jobs), int((~not_bn).sum()), dict(pd.Series(lang).value_counts().head(5))))
    wavs = [w for w, k in zip(wavs, not_bn) if k]; jpgs = [j for j, k in zip(jpgs, not_bn) if k]
    spk = np.array(['vox_' + s for s, k in zip(spk, not_bn) if k]); lang = lang[not_bn]
    feats = {'spk': spk, 'lang': lang,
             'gender': np.array([VOX_GENDER.get(s[4:], '?') for s in spk]),
             'clip': enc_clip(jpgs).astype(np.float32),
             'ecapa': enc_ecapa(wavs).astype(np.float32),
             'wavlm_sv': enc_wavlm(wavs).astype(np.float32)}
    np.savez(out_path, **feats)
    shutil.rmtree(W, ignore_errors=True)
    note('%s: saved %d clips of %d people in %.0f min total' % (tag, len(spk), len(set(spk)), (time.time() - t0) / 60))

VOX = [dict(np.load(os.path.join(FEATDIR, f))) for f in sorted(os.listdir(FEATDIR)) if f.startswith('vox_') and f.endswith('.npz')]
VOXALL = {k: np.concatenate([v[k] for v in VOX]) for k in ['spk', 'clip', 'ecapa', 'wavlm_sv']} if VOX else None
note('VoxCeleb2 available: %d clips, %d people' % ((len(VOXALL['spk']), len(set(VOXALL['spk']))) if VOX else (0, 0)))

In [ ]:
# Cell 7 - (training part) skipped when RUN_TRAINING = False -> continue with Notebook 08
if not RUN_TRAINING:
    print('=' * 70)
    print('SUMMARY  (copy everything between the lines and paste to Claude)')
    print('=' * 70)
    for s in SUMMARY:
        print(s)
    print('=' * 70)
    raise SystemExit('Extraction finished. Now run Notebook 08 (training with the domain fix).')

# combined table: challenge rows, then MAV-Celeb, then VoxCeleb2 + model + validation (as Notebook 06)
import torch.nn as nn
import torch.nn.functional as Fn
from sklearn.metrics import roc_curve

def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def face_of(clip):
    return l2(clip).astype(np.float32)
def voice_of(ecapa, wavlm):
    return np.concatenate([l2(ecapa), l2(wavlm)], 1).astype(np.float32)

N_CH, N_MAV = len(y), len(MAV['spk'])
ext_spk = list(MAV['spk']) + (list(VOXALL['spk']) if VOXALL else [])
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.concatenate([y, np.array([ext_lab[s] for s in ext_spk])])
parts_f = [face_of(FEAT['clip']['train']), face_of(MAV['clip'])]
parts_v = [voice_of(FEAT['ecapa']['train'], FEAT['wavlm_sv']['train']), voice_of(MAV['ecapa'], MAV['wavlm_sv'])]
if VOXALL:
    parts_f.append(face_of(VOXALL['clip'])); parts_v.append(voice_of(VOXALL['ecapa'], VOXALL['wavlm_sv']))
XF = torch.tensor(np.concatenate(parts_f), device=dev); XV = torch.tensor(np.concatenate(parts_v), device=dev)
EXT_ROWS = {'mav': N_CH + np.arange(N_MAV),
            'mav+vox': N_CH + np.arange(len(ext_spk))}

class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv)
        return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv)
        a = torch.sigmoid(self.att(torch.cat([f, v], 1)))
        fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused)
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t()
    same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm
    ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()   # HIGHER = same person
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed)
    spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5
rng = np.random.default_rng(42)
folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k])
    va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))
print('Combined table:', XF.shape, '| people:', len(set(Y_ALL)))

In [ ]:
# Cell 8 - train E2 (challenge + MAV-Celeb) vs E3 (+ VoxCeleb2)  (RESUMABLE)
SEEDS = [1, 2, 3]
P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS = 16, 8, 2.0, 1e-4, 100, 15, 60

def train_model(tr_idx, vsets=None, seed=1, max_epochs=MAX_EP):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    classes = np.unique(Y_ALL[tr_idx]); remap = {c: i for i, c in enumerate(classes)}
    ytr_np = np.array([remap[c] for c in Y_ALL[tr_idx]]); ytr = torch.tensor(ytr_np, device=dev)
    tr = torch.tensor(tr_idx, device=dev)
    norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6)   # TRAIN stats only
    mf, sf_, mv, sv = norm
    model = FOP(len(classes), XF.shape[1], XV.shape[1]).to(dev); model.norm = norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01)
    pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(classes))]
    best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(classes), P_ID, replace=False)
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv)
            loss = Fn.cross_entropy(logits, ytr[j]) + ALPHA * opl_loss(fused, ytr[j])
            opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[F], XV[V])) for V, F, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep

N_VOX_PEOPLE = len(set(VOXALL['spk'])) if VOXALL else 0
VARIANTS = {'E2_mav': 'mav', 'E3_mav+vox%d' % N_VOX_PEOPLE: 'mav+vox'} if VOXALL else {'E2_mav': 'mav'}
RESDIR = os.path.join(PROJECT, 'results'); os.makedirs(RESDIR, exist_ok=True)
CSV = os.path.join(RESDIR, 'nb07_cv.csv')
done = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame(columns=['variant', 'seed', 'fold'])
t0 = time.time()
for name, ext_key in VARIANTS.items():
    for seed in SEEDS:
        for k in range(K):
            if ((done.variant == name) & (done.seed == seed) & (done.fold == k)).any():
                continue
            tr_idx, vsets = FOLD_DATA[k]
            model, best_ep = train_model(np.concatenate([tr_idx, EXT_ROWS[ext_key]]), vsets, seed=seed)
            e_std, e_gen = [eer(L, same_score(model, XF[F], XV[V])) for V, F, L in vsets]
            row = pd.DataFrame([dict(variant=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
            row.to_csv(CSV, mode='a', header=not os.path.exists(CSV), index=False)
            done = pd.concat([done, row])
            print('%-18s seed %d fold %d | ep %3d | std %.2f | gender %.2f | %.0f min'
                  % (name, seed, k, best_ep, e_std, e_gen, (time.time() - t0) / 60))

R = pd.read_csv(CSV); R = R[R.variant.isin(VARIANTS)]
R['overall'] = (R['std'] + R['gender']) / 2
per_seed = R.groupby(['variant', 'seed'])[['std', 'gender', 'overall']].mean().reset_index()
tab = per_seed.groupby('variant').agg(std=('std', 'mean'), gender=('gender', 'mean'),
                                      overall=('overall', 'mean'), seed_sd=('overall', 'std'))
tab['gap'] = tab['gender'] - tab['std']
tab = tab.loc[[v for v in VARIANTS if v in tab.index]].round(2)
note('MORE PEOPLE, ROUND 2 (K10 features, validation on held-out CHALLENGE people, 5-fold x 3 seeds)')
note(tab.to_string())

In [ ]:
# Cell 9 - final model for the best variant (all 70 challenge people + external, 3 seeds) -> submission ZIP
LOWER_IS_SAME = True
LANG_FILE = {'English': 'sub_score_v4_English_heard.txt', 'Bangla': 'sub_score_v4_Bangla_unheard.txt'}
BEST = tab['overall'].idxmin()
ep = int(np.median(R[R.variant == BEST]['best_ep']))
tr_all = np.concatenate([np.arange(N_CH), EXT_ROWS[VARIANTS[BEST]]])
finals = [train_model(tr_all, None, seed=s, max_epochs=ep)[0] for s in SEEDS]
OUT = '/content/sub_07_' + BEST
for track in ['no_gender', 'gender']:
    os.makedirs(os.path.join(OUT, track), exist_ok=True)
    for lang, fname in LANG_FILE.items():
        split = track + '_' + lang
        pair_ids = [l.split()[0] for l in open(find_one('/' + track + '/', lang + '_test.txt')) if l.strip()]
        xf = torch.tensor(face_of(FEAT['clip'][split]), device=dev)
        xv = torch.tensor(voice_of(FEAT['ecapa'][split], FEAT['wavlm_sv'][split]), device=dev)
        s = np.mean([same_score(m, xf, xv) for m in finals], axis=0)
        out = -s if LOWER_IS_SAME else s
        with open(os.path.join(OUT, track, fname), 'w') as f:
            for pid, v in zip(pair_ids, out):
                f.write('%s %.6f\n' % (pid, v))
SUBDIR = os.path.join(PROJECT, 'submissions'); os.makedirs(SUBDIR, exist_ok=True)
ZIP_PATH = os.path.join(SUBDIR, '07_' + BEST.replace('+', '_') + '.zip')
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for track in ['no_gender', 'gender']:
        for fname in LANG_FILE.values():
            zf.write(os.path.join(OUT, track, fname), arcname=track + '/' + fname)
note('BEST: %s -> submission ZIP saved: %s (%d epochs)' % (BEST, ZIP_PATH[len(BASE):], ep))

LOG = os.path.join(PROJECT, 'experiments.csv')
log = pd.read_csv(LOG) if os.path.exists(LOG) else pd.DataFrame()
if len(log) and 'run' in log:
    log = log[~log.run.isin(['07_' + v for v in tab.index])]
new = pd.DataFrame([{'date': time.strftime('%Y-%m-%d %H:%M'), 'run': '07_' + v, 'change': 'K10 + external rows: %s' % VARIANTS[v],
                     'seed': '1,2,3', 'cv_std': tab.loc[v, 'std'], 'cv_gender': tab.loc[v, 'gender'],
                     'cv_overall': tab.loc[v, 'overall'], 'codabench_dev': ''} for v in tab.index])
pd.concat([log, new], ignore_index=True).to_csv(LOG, index=False)
print('=' * 70)
print('SUMMARY  (copy everything between the lines and paste to Claude)')
print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)